In [ ]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/deep-learning-course-project')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import get_cifar10_loaders_and_splits
from src.utils.training import calculate_accuracy, load_weights
from src.models.architectures.RestNet18 import *
from src.models.architectures.ScatNet18 import *


from kornia import augmentation as K

In [ ]:
DEBUG = True
EXP_NAME = "models_robustness"
print(f"Starting experiment {EXP_NAME}. DEBUG={DEBUG}")

n_samples_per_class_train = [10, 50, 100, 500, 1000, 4000]

In [4]:
# Get Augmentations
augmentatios = {
    'gray': K.RandomGrayscale(same_on_batch=True, p=1.0, keepdim=True).to(DEVICE),
    'hor_flip': K.RandomHorizontalFlip(p=1.0, same_on_batch=True, keepdim=True).to(DEVICE),
    'gauss_noise': K.RandomGaussianNoise(mean=0.0, std=0.1, same_on_batch=True, keepdim=True).to(DEVICE),
    'gauss_blur': K.RandomGaussianBlur(kernel_size=(5,5), sigma=(0.1,2.0), p=1.0, same_on_batch=True, keepdim=True).to(DEVICE),
    'median_blur': K.RandomMedianBlur(kernel_size=(3,3), p=1.0, same_on_batch=True, keepdim=True).to(DEVICE)
}

In [ ]:
# Get models
resnet_models = {}
scatnet_models = {}
for n_samples in n_samples_per_class_train:
    resnet_models[n_samples] = load_weights(
        MakeResNet18(),
        experiment_name="baseline_acc_vs_n_samples",
        model_name=f"ResNet18_{n_samples}",
        device=DEVICE
    )
    scatnet_models[n_samples] = load_weights(
        MakeScatNet(L=10),
        experiment_name="scatnet_acc_vs_n_samples",
        model_name=f"ScatNet18_{n_samples}",
        device=DEVICE
    )

In [ ]:
# Get test loader
_, _, testloader, _, _, test_set = get_cifar10_loaders_and_splits()

In [ ]:
# Visualize augmentations
if not DEBUG:
    img, label = test_set[0]

    # img is (C, H, W) tensor, needs to be (H, W, C) for matplotlib
    img_np = img.permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(1, len(augmentations) + 1, figsize=(3 * (len(augmentations) + 1), 3))

    # Show original
    axes[0].imshow(img_np)
    axes[0].set_title("Original")
    axes[0].axis('off')

    # Show each augmentation
    for i, (aug_name, aug) in enumerate(augmentations.items()):
        # Add batch dimension, apply aug, remove batch dimension
        aug_img = aug(img.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()
        aug_img_np = aug_img.permute(1, 2, 0).numpy()
        
        # Clip values to [0, 1] for display (noise can push values outside)
        aug_img_np = aug_img_np.clip(0, 1)
        
        axes[i + 1].imshow(aug_img_np)
        axes[i + 1].set_title(aug_name)
        axes[i + 1].axis('off')

    plt.suptitle(f"Test Image (Label: {label})", fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / "augmentations_visualization.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Check models accuracy on each augmentation
resnet_accs = {}
scatnet_accs = {}

if not DEBUG:
    # Resnet
    for n_samples in n_samples_per_class_train:
        resnet_accs[n_samples] = {}
        model = resnet_models[n_samples]
        for aug_name, aug in augmentations.items():
            resnet_accs[n_samples][aug_name] = calculate_accuracy(
                model, testloader, DEVICE, augmentations=aug
            )

    # ScatNet
    for n_samples in n_samples_per_class_train:
        scatnet_accs[n_samples] = {}
        model = scatnet_models[n_samples]
        for aug_name, aug in augmentations.items():
            scatnet_accs[n_samples][aug_name] = calculate_accuracy(
                model, testloader, DEVICE, augmentations=aug
            )

In [ ]:
# Plot results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

aug_names = list(augmentations.keys())
n_samples_list = list(n_samples_per_class_train)

for i, aug_name in enumerate(aug_names):
    ax = axes[i]
    
    # Get accuracies for each n_samples
    resnet_acc_list = [resnet_accs[n][aug_name] for n in n_samples_list]
    scatnet_acc_list = [scatnet_accs[n][aug_name] for n in n_samples_list]
    
    # Plot
    ax.plot(n_samples_list, resnet_acc_list, 'o-', label='ResNet18', color='blue', markersize=8)
    ax.plot(n_samples_list, scatnet_acc_list, 's-', label='ScatNet18', color='orange', markersize=8)
    
    ax.set_xlabel('Samples per Class')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(f'{aug_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')  # Log scale for n_samples if they vary widely

# Hide unused subplots if any
for j in range(len(aug_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Model Robustness: ResNet18 vs ScatNet18', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / "robustness_comparison.png", dpi=150, bbox_inches='tight')
plt.show()